# P05: Applications & Modern Python

Functions let you name and reuse an analysis. We will write a real one --
signal-detection *d'* -- and use it to expose two of the most important hazards in
scientific code: a formula that returns `inf` at the edges, and Python's notorious
mutable-default-argument bug.

## A function for d-prime (sensitivity)

In [1]:
from scipy.stats import norm

def dprime(hits, misses, false_alarms, correct_rejections):
    """Signal-detection d': sensitivity in z-units."""
    hit_rate = hits / (hits + misses)
    fa_rate = false_alarms / (false_alarms + correct_rejections)
    return norm.ppf(hit_rate) - norm.ppf(fa_rate)

print(round(dprime(45, 5, 10, 40), 3))

2.123


## What happens at the edges?

In [2]:
# a 'perfect' subject: hit_rate = 1.0  ->  norm.ppf(1.0) = +infinity
print(dprime(50, 0, 10, 40))   # inf -- silently broken, no error raised

inf


In [3]:
def dprime_corrected(hits, misses, false_alarms, correct_rejections):
    # log-linear correction (Hautus, 1995): add 0.5 to each cell and 1 to each total
    hit_rate = (hits + 0.5) / (hits + misses + 1)
    fa_rate = (false_alarms + 0.5) / (false_alarms + correct_rejections + 1)
    return norm.ppf(hit_rate) - norm.ppf(fa_rate)

print(round(dprime_corrected(50, 0, 10, 40), 3))   # a finite, usable value

3.155


:::{admonition} Human check — the naive d' formula is 'functional but incorrect'
:class: warning
The first `dprime` is exactly the kind of code an AI writes confidently: it matches
the textbook formula and runs without error. But any subject with a perfect hit
rate (or zero false alarms) yields `inf`, which then poisons every average,
*t*-test, or plot downstream -- often showing up only as a weird-looking figure days
later. Whether to apply a correction (and which one) is a **scientific** decision a
human must make; the computer cannot know that `inf` is meaningless here.
:::

## The mutable default argument bug

In [4]:
# DANGER: a default list is created ONCE, when the function is defined,
# and then SHARED across every call that doesn't pass its own list
def add_trial(rt, collected=[]):          # <-- the bug is right here
    collected.append(rt)
    return collected

print(add_trial(500))   # [500]
print(add_trial(600))   # [500, 600]  -- last call's data came back!

[500]
[500, 600]


In [5]:
# the fix: use None as the default and create a fresh list inside
def add_trial(rt, collected=None):
    if collected is None:
        collected = []
    collected.append(rt)
    return collected

print(add_trial(500))   # [500]
print(add_trial(600))   # [600]  -- independent, as expected

[500]
[600]


:::{admonition} Human check — mutable defaults silently accumulate state
:class: warning
`def f(x, cache=[])` and `def f(x, opts={})` look harmless and run perfectly in a
quick test. But the default object persists between calls, so results from one
participant can leak into the next. Always use `None` as the sentinel and build the
container inside the function.
:::

:::{admonition} Modern Python
:class: tip
Modern function signatures often add **type hints** and **keyword-only arguments**
(everything after a bare `*` must be passed by name), which makes call sites
self-documenting:

```python
def dprime(hits: int, misses: int, *, correction: bool = True) -> float:
    ...
```
:::